# Danh — adapter ở Stage 1/2/3 của SVTRv2-T, thử đủ 7 tổ hợp, Kaggle 2×T4

**Mục tiêu.** Kiểm giả thuyết của nhóm: module nhỏ sửa feature sau từng stage có thể tăng khả năng đọc motion blur mà không thay trọng số SVTRv2. Bảy tổ hợp là `1`, `2`, `3`, `1+2`, `1+3`, `2+3`, `1+2+3`. Tên số là **sau** Stage tương ứng. Mỗi module giữ nguyên kích thước feature, cộng phần hiệu chỉnh bằng residual và bắt đầu đúng hàm đồng nhất.

**Nguồn và ranh giới khẳng định.** Encoder/decoder, checkpoint, CTC và tiền xử lý dùng [OpenOCR/SVTRv2](https://github.com/Topdu/OpenOCR), commit `1ccfc6ee6161f7133b192e16af3a9265273997ba`. MotionBlurAug lấy đúng từ notebook `aug-svtrv2.ipynb` của nhóm. Residual bottleneck với tích chập theo chiều ngang là **thiết kế thử nghiệm của nhóm**, gợi ý từ dòng adapter tích chập; chưa có paper chứng minh chính module này tối ưu cho motion blur STR. Không gọi đây là reproduction của Convpass/KD-LTR. Module mới chỉ được train trên **Union14M medium + MotionBlurAug**, nhãn CTC gốc. Cả patch embedding, 3 stage và RCTC đều cố định.

**Phép so.** B0 checkpoint gốc, S2_AUG checkpoint đã train trước (Stage 2+3+RCTC, 12 epoch), và 7 adapter từ B0. Thử nhanh 7 tổ hợp cùng 2 epoch, chọn 2 tổ hợp theo validation IC13 ảnh sạch và blur tổng hợp, rồi **train lại từ B0 đủ 12 epoch** cho hai tổ hợp đó. Chỉ đánh giá IIIT5K test khi mô hình cuối đã cố định. Vì 7 tổ hợp được xem nhiều lần trên IC13, đây vẫn là nghiên cứu thăm dò; xác nhận cuối cần tập test mới.

**Giới hạn công bằng.** Adapter và S2_AUG cùng dữ liệu augmentation, nhưng khác số tham số train được và cơ chế tối ưu. Đây là phép so phương pháp ở ngân sách 12 epoch, không tách riêng nguyên nhân từ số tham số. S2_AUG cần gắn output Kaggle cũ chứa `S2_AUG/latest.pth`; gói `audit_only.zip` không có checkpoint. Không tự thay bằng một checkpoint tên gần giống.


## 0. Inputs — sửa hai đường dẫn nếu Kaggle đặt tên dataset khác

Bật **2×T4** và Internet. Add bộ IIIT5K blur của nhóm, và **output bản chạy S2_AUG** chứa `S2_AUG/latest.pth` vào Kaggle Input. Dữ liệu Union14M sẽ được lấy như notebook S2 trước. Nếu output S2_AUG không còn, cần chạy lại notebook S2 trước để tạo checkpoint; không thể phục hồi trọng số từ `audit_only.zip`.


In [ ]:
BLUR_DATA_HINT='/kaggle/input/datasets/nguynthnhthy/data-blur'
S2_AUG_CKPT=''  # /kaggle/input/<output-s2>/S2_AUG/latest.pth; để rỗng cho tự tìm
LOCAL_TINY_CKPT=''; LOCAL_TINY_CFG=''
BATCH,ACCUM=48,1
TRAIN_PREF=['medium','normal']; MAX_TRAIN_GB=8; TRAIN_LMDB_OVERRIDE=[]
RESTORED_CAM=False; EXTRA_KINDS=[]; SEED=1234
EXP=dict(family='svtrv2',aug=True,aug_p=.5,len_scale=1.,angle_mode='protocol',epochs=12,lr=2e-4,warmup_frac=.075)
PILOT_EPOCHS=2; FINAL_EPOCHS=12; TOP_K=2
assert TOP_K==2 and PILOT_EPOCHS<FINAL_EPOCHS


In [ ]:
import os, sys, io, re, glob, time, json, random, subprocess, shutil, zipfile, tarfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
WORK = Path('/kaggle/working/adapter_grid_2026_v1') if Path('/kaggle').exists() else Path('work/adapter_grid_2026_v1'); WORK.mkdir(exist_ok=True, parents=True)
OPENOCR = WORK/'OpenOCR'
if not OPENOCR.exists():
    subprocess.run(['git','clone','https://github.com/Topdu/OpenOCR.git',str(OPENOCR)], check=True)
subprocess.run(['git','-C',str(OPENOCR),'checkout','1ccfc6ee6161f7133b192e16af3a9265273997ba'],check=True)
assert subprocess.check_output(['git','-C',str(OPENOCR),'rev-parse','HEAD'],text=True).strip()=='1ccfc6ee6161f7133b192e16af3a9265273997ba'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(OPENOCR/'requirements.txt')], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--only-binary',':all:','pygame'], check=False)   # KHÔNG ghim 2.5.2: không có wheel cho Python mới
subprocess.run([sys.executable,'-m','pip','install','-q','lmdb','gdown','huggingface_hub','scipy','tabulate'], check=False)
# ---- VÁ tương thích (Python 3.13 / numpy 2 / pygame 2.6 trên Kaggle) ----
import importlib.util
_spec = importlib.util.find_spec('imgaug')            # không import imgaug (đang hỏng với numpy 2)
if _spec:
    _root = Path(_spec.origin).parent
    _rep = [(r'np\.sctypes\["float"\]', '[np.float16, np.float32, np.float64]'),
            (r'np\.sctypes\["int"\]',   '[np.int8, np.int16, np.int32, np.int64]'),
            (r'np\.sctypes\["uint"\]',  '[np.uint8, np.uint16, np.uint32, np.uint64]'),
            (r'np\.complex(?![_0-9A-Za-z])', 'np.complex128'), (r'np\.bool(?![_0-9A-Za-z])', 'np.bool_'),
            (r'np\.float(?![_0-9A-Za-z])', 'np.float64'),     (r'np\.int(?![_0-9A-Za-z])', 'np.int64')]
    _n = 0
    for _f in _root.rglob('*.py'):
        _s = _f.read_text(); _t = _s
        for _a, _b in _rep: _t = re.sub(_a, _b, _t)
        if _t != _s: _f.write_text(_t); _n += 1
    print('vá imgaug:', _n, 'file')
_p = OPENOCR/'openrec/preprocess/cam_label_encode.py'
_s = _p.read_text(); _p.write_text(_s.replace('pygame.locals.SRCALPHA', 'pygame.SRCALPHA')); print('vá cam_label_encode (pygame.SRCALPHA)')
# kiểm tra các import mà train/eval của OpenOCR cần, trong tiến trình con (đúng môi trường lúc train)
for mod in ['pygame.freetype','imgaug','lmdb','cv2','rapidfuzz','pyclipper']:
    r = subprocess.run([sys.executable,'-c',f'import {mod}'], capture_output=True, text=True)
    print(('OK   ' if r.returncode==0 else 'LỖI ')+mod, '' if r.returncode==0 else r.stderr.strip().splitlines()[-1])
    assert r.returncode == 0, f'Thiếu/hỏng thư viện {mod}: xem dòng LỖI ở trên.'
os.chdir(OPENOCR); sys.path.insert(0, str(OPENOCR))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Bật GPU trong Settings > Accelerator.'
# CAM vẽ mask glyph bằng font; bản gốc dùng Arial. Không có thì dùng Liberation Sans (GHI RÕ trong báo cáo). Chỉ cần cho họ CAM.
FONT_USED = None
cands_f = glob.glob('/usr/share/fonts/**/arial.ttf', recursive=True) or glob.glob('/usr/share/fonts/**/LiberationSans-Regular.ttf', recursive=True)
if cands_f: shutil.copy(cands_f[0], OPENOCR/'arial.ttf'); FONT_USED = cands_f[0]
if EXP['family'] == 'cam': assert cands_f, 'Không có font. Tải arial.ttf đặt vào thư mục OpenOCR/.'
print('font mask:', FONT_USED)

# vá torch.load trong OpenOCR: torch>=2.6 mặc định weights_only=True làm hỏng nạp/resume checkpoint
_npt = 0
for _f in list(OPENOCR.glob('tools/**/*.py')) + list(OPENOCR.glob('openrec/**/*.py')):
    _s = _f.read_text(); _t = re.sub(r'torch\.load\(([^()\n]*?)\)', lambda m: m.group(0) if 'weights_only' in m.group(1) else 'torch.load(' + m.group(1) + ', weights_only=False)', _s)
    if _t != _s: _f.write_text(_t); _npt += 1
print('vá torch.load(weights_only=False):', _npt, 'file')


In [ ]:
assert torch.cuda.device_count()==2, 'Bật 2×T4 trước khi train'; print([torch.cuda.get_device_name(i) for i in range(2)])

In [ ]:
# Fail early if the reference checkpoint is absent, before hours of training.
if not S2_AUG_CKPT:
 _s2_hits=list(Path('/kaggle/input').rglob('S2_AUG/latest.pth'))
 assert len(_s2_hits)==1, f'Add exactly one previous Kaggle S2_AUG output, found {_s2_hits}. Or set S2_AUG_CKPT explicitly.'
 S2_AUG_CKPT=str(_s2_hits[0])
assert Path(S2_AUG_CKPT).is_file(),S2_AUG_CKPT
print('S2 reference:',S2_AUG_CKPT)


## 1. Dữ liệu và kiểm tra ảnh/nhãn

Các cell dưới đây dùng phần chuẩn bị IIIT5K và Union14M đã chạy của notebook S2 trước. Chúng kiểm nguồn IIIT5K sạch, nối ID của ảnh blur với nhãn, số ảnh mỗi mức và tạo LMDB. Nếu ID/nhãn không khớp, notebook dừng. **Ảnh IIIT5K test không được đưa vào train hay chọn tổ hợp.** Dải blur train theo manifest của nhóm là một giới hạn đã biết và được ghi vào audit.


In [ ]:
def find_blur_root():
    if BLUR_DATA_HINT:
        hit = glob.glob(f'{BLUR_DATA_HINT}/**/degradation_manifest*', recursive=True)
        assert hit, f'Không thấy degradation_manifest* dưới {BLUR_DATA_HINT}'
        return Path(hit[0]).parent
    for base in ['/kaggle/input', '/kaggle/working', '.']:
        hit = glob.glob(f'{base}/**/degradation_manifest*', recursive=True)
        if hit: return Path(hit[0]).parent
    for base in ['/kaggle/input', '/kaggle/working', '.']:
        z = glob.glob(f'{base}/**/IIIT5K_Blur_Data*.zip', recursive=True)
        if z:
            out = WORK/'blur_data'
            if not out.exists(): zipfile.ZipFile(z[0]).extractall(out)
            hit = glob.glob(f'{out}/**/degradation_manifest*', recursive=True)
            if hit: return Path(hit[0]).parent
    raise FileNotFoundError('Không thấy dữ liệu blur. Add dataset vào notebook hoặc đặt BLUR_DATA_HINT.')
BROOT = find_blur_root(); print('dữ liệu blur:', BROOT)
man_path = sorted(glob.glob(str(BROOT/'degradation_manifest*')))[0]
man = pd.read_csv(man_path) if man_path.endswith('.csv') else pd.read_excel(man_path)
print(man_path, man.shape); print(list(man.columns)); display(man.head(6))

In [ ]:
# Tự đoán cột theo tên; SỬA tay nếu in ra sai.
def pick(keys):
    for c in man.columns:
        if any(k in c.lower() for k in keys): return c
    return None
COLS = dict(file=pick(['file','path','image','name']), label=pick(['label','text','gt','word','transcr']),
            kind=pick(['type','kind','blur','degrad']), level=pick(['level','sever','strength','intens']))
print('COLS =', COLS)
for k in ('kind','level'):
    if COLS[k]: print(k, '->', man[COLS[k]].astype(str).value_counts().to_dict())

In [ ]:
import lmdb, scipy.io
IMG_EXT = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
imgs = {p.name: p for p in (BROOT/'blur').rglob('*') if p.suffix.lower() in IMG_EXT}
print('số ảnh blur:', len(imgs))
def img_id(name):
    m = re.search(r'iiit_test_(\d+)', name); return int(m.group(1)) if m else None
def tail_of(name):
    t = name.split('__', 1)[1] if '__' in name else ''; return os.path.splitext(t)[0]
tails = {}
for n in imgs: tails[tail_of(n)] = tails.get(tail_of(n), 0) + 1
print('các điều kiện blur (phần đuôi tên file) và số ảnh:')
for t, c in sorted(tails.items()): print(f'   {t!r}: {c}')

def kind_of(s):
    s = s.lower()
    if 'gauss' in s: return 'gaussian'
    if any(w in s for w in ['defocus','disk','circ','lens','bokeh']): return 'defocus'
    if any(w in s for w in ['motion','linear']): return 'motion'
    if any(w in s for w in ['clean','orig']): return 'clean'
    return None
def level_of(s):
    s = s.lower()
    for k, v in [('mild','mild'),('med','medium'),('sev','severe'),('serve','severe'),('strong','severe'),('low','mild'),('mid','medium'),('high','severe')]:
        if k in s: return v
    return None
kinds = {t: kind_of(t) for t in tails}
unk = [t for t in tails if kinds[t] is None]
assert not unk, f'Không nhận diện được loại blur cho các đuôi: {unk}. Gửi danh sách ở trên để chỉnh kind_of.'
tail_cond = {}
for k in set(kinds.values()):
    ts = [t for t in tails if kinds[t] == k]; lv = {t: level_of(t) for t in ts}
    if k == 'clean': tail_cond.update({t: 'clean' for t in ts}); continue
    if None in lv.values() or len(set(lv.values())) != len(ts):
        assert len(ts) == 3, f'{k}: {len(ts)} đuôi, không gán được mild/medium/severe: {ts}'
        order = sorted(ts, key=lambda t: [int(x) if x.isdigit() else x for x in re.split(r'(\d+)', t)])
        lv = dict(zip(order, ['mild','medium','severe']))
        print('CẢNH BÁO: không đọc được mức từ tên file; gán theo thứ tự tự nhiên', k, lv, '-> KIỂM TRA lại với tham số blur thật')
    tail_cond.update({t: f'{k}_{lv[t]}' for t in ts})
print('ánh xạ đuôi -> điều kiện:', tail_cond)

# ---- ảnh sạch + nhãn: lấy từ nguồn IIIT5K (thư mục blur/ không có ảnh sạch) ----
def read_lmdb_pairs(path):
    env = lmdb.open(str(path), readonly=True, lock=False, readahead=False); out = []
    with env.begin() as t:
        total = int(t.get(b'num-samples')); base = 1 if t.get(b'image-%09d' % 1) is not None else 0
        for i in range(base, base+total):
            ib, lb = t.get(b'image-%09d' % i), t.get(b'label-%09d' % i)
            out.append((Image.open(io.BytesIO(ib)).convert('RGB'), lb.decode('utf-8','ignore')))
    env.close(); return out

def load_clean_source():
    roots = [BROOT, BROOT.parent] + ([Path(BLUR_DATA_HINT)] if BLUR_DATA_HINT else [])
    mats = [m for r in roots for m in r.rglob('testdata.mat')]
    if not mats:
        tars = [p for r in roots for p in r.rglob('*') if p.is_file() and re.search(r'\.tar(\.\w+)?$', p.name)]
        print('tar tìm thấy:', tars)
        if tars:
            ex = WORK/'iiit5k_src'
            if not ex.exists(): tarfile.open(tars[0]).extractall(ex)
            mats = list(ex.rglob('testdata.mat'))
    if mats:
        mat = mats[0]; print('nguồn sạch: testdata.mat ở', mat)
        td = scipy.io.loadmat(mat, squeeze_me=True, struct_as_record=False)['testdata']
        out = []
        for nm, gt in zip(td.ImgName, td.GroundTruth):
            p = mat.parent/str(nm)
            if not p.exists(): p = mat.parent/'test'/os.path.basename(str(nm))
            out.append((Image.open(p).convert('RGB'), str(gt)))
        return out
    print('không có tar/mat trong dataset; dùng IIIT5k LMDB từ Hugging Face (topdu/OpenOCR-Data/evaluation/IIIT5k)')
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='topdu/OpenOCR-Data', repo_type='dataset', local_dir=WORK/'hf_eval', allow_patterns=['evaluation/IIIT5k/*'])
    return read_lmdb_pairs(WORK/'hf_eval'/'evaluation'/'IIIT5k')

clean_src = load_clean_source(); print('ảnh sạch nguồn:', len(clean_src))
CLEAN_DIR = WORK/'clean_png'; CLEAN_DIR.mkdir(exist_ok=True)
clean_paths = []
for i, (im, lb) in enumerate(clean_src):
    p = CLEAN_DIR/f'iiit_test_{i:05d}__clean.png'
    if not p.exists(): im.save(p)
    clean_paths.append(p)

# ---- kiểm tra: id ảnh blur có khớp thứ tự ảnh sạch không? (đối sánh ảnh mờ nhẹ nhất với ảnh sạch) ----
def feat(p):
    a = np.asarray(Image.open(p).convert('L').resize((64, 16), Image.BILINEAR), np.float32).ravel()
    a = a - a.mean(); return a / (np.linalg.norm(a) + 1e-6)
chk = sorted([c for c in set(tail_cond.values()) if c != 'clean'], key=lambda c: (not c.endswith('mild'), c))[0]
chk_tail = [t for t, c in tail_cond.items() if c == chk][0]
chk_items = sorted([(img_id(n), p) for n, p in imgs.items() if tail_of(n) == chk_tail])
ids = [i for i, _ in chk_items]; assert max(ids) < len(clean_paths), f'id lớn nhất {max(ids)} >= số ảnh sạch {len(clean_paths)}'
Xb = np.stack([feat(p) for _, p in chk_items]); Xc = np.stack([feat(p) for p in clean_paths])
nn_idx = (Xb @ Xc.T).argmax(1); agree = float(np.mean(nn_idx == np.array(ids)))
print(f'kiểm tra khớp id ({chk}): {agree*100:.1f}% ảnh mờ giống nhất đúng ảnh sạch cùng chỉ số')
if agree >= 0.95: id2lab = {i: clean_src[i][1] for i in range(len(clean_src))}
else:
    print('CẢNH BÁO: id không khớp thứ tự nguồn -> lấy nhãn theo ảnh sạch giống nhất (đối sánh)')
    id2lab = {i: clean_src[j][1] for i, j in zip(ids, nn_idx)}
assert agree > 0.5, 'Ảnh blur và ảnh nguồn gần như không tương ứng; kiểm tra lại nguồn IIIT5K.'

GROUPS = {'clean': [(clean_paths[i], clean_src[i][1]) for i in sorted(id2lab)]}
for n, p in imgs.items():
    c = tail_cond[tail_of(n)]
    if c != 'clean': GROUPS.setdefault(c, []).append((p, id2lab.get(img_id(n))))
print({c: len(v) for c, v in sorted(GROUPS.items())})
assert all(f'motion_{l}' in GROUPS for l in ['mild','medium','severe']), 'Thiếu motion_mild/medium/severe: xem ánh xạ đuôi -> điều kiện ở trên.'
assert all(lb is not None for v in GROUPS.values() for _, lb in v), 'Có ảnh không có nhãn.'

In [ ]:
def write_lmdb(items, out):
    Path(out).mkdir(parents=True, exist_ok=True); env = lmdb.open(str(out), map_size=1<<33); n = 0
    with env.begin(write=True) as t:
        for p, lb in sorted(items, key=lambda x: x[0].name):
            try: im = Image.open(p).convert('RGB')
            except Exception: continue
            n += 1; b = io.BytesIO(); im.save(b, 'PNG')
            t.put(b'image-%09d' % n, b.getvalue()); t.put(b'label-%09d' % n, lb.encode())
        t.put(b'num-samples', str(n).encode())
    env.close(); return n
LM = WORK/'blur_lmdb'; N = {}
for cond, items in GROUPS.items():
    d = LM/cond/'test'
    N[cond] = int(open(d/'n.txt').read()) if (d/'n.txt').exists() else write_lmdb(items, d)
    if not (d/'n.txt').exists(): open(d/'n.txt','w').write(str(N[cond]))
print(N)
CONDS = ['clean'] + [f'motion_{l}' for l in ['mild','medium','severe']] + [f'{k}_{l}' for k in EXTRA_KINDS for l in ['mild','medium','severe'] if f'{k}_{l}' in GROUPS]
print('sẽ đánh giá:', CONDS)

In [ ]:
TRAIN_ROOT = WORK/'train_root'
picked = []
if RESTORED_CAM:
    print('đã khôi phục checkpoint CAM -> bỏ qua tải dữ liệu train')
else:
    if TRAIN_LMDB_OVERRIDE:
        picked = [Path(p) for p in TRAIN_LMDB_OVERRIDE]
    else:
        from huggingface_hub import HfApi, snapshot_download
        REPO = 'topdu/OpenOCR-Data'; api = HfApi()
        files = api.list_repo_files(REPO, repo_type='dataset')
        mdb = [f for f in files if f.endswith('data.mdb')]
        tl = [f for f in mdb if 'train' in f.lower() and not f.lower().startswith(('evaluation','u14m'))]
        print('thư mục LMDB train tìm thấy:', [os.path.dirname(f) for f in tl][:20])
        assert tl, 'Repo HF không có data.mdb train. Điền TRAIN_LMDB_OVERRIDE hoặc build LMDB theo docs/svtrv2.md.'
        info = {i.path: i.size for i in api.get_paths_info(REPO, tl, repo_type='dataset')}
        ok = {f: s_ for f, s_ in info.items() if s_ and s_/1e9 <= MAX_TRAIN_GB}
        assert ok, f'Mọi thư mục train > {MAX_TRAIN_GB} GB: {({os.path.dirname(f): round(s_/1e9,1) for f, s_ in info.items()})}'
        def rank(f):
            for i, k in enumerate(TRAIN_PREF):
                if k in f.lower(): return (i, ok[f])
            return (len(TRAIN_PREF), ok[f])
        best = sorted(ok, key=rank)[0]; d = os.path.dirname(best)
        print('chọn:', d, f'({ok[best]/1e9:.2f} GB)')
        if not (Path(WORK/'hf_data')/d/'data.mdb').exists():
            snapshot_download(repo_id=REPO, repo_type='dataset', local_dir=WORK/'hf_data', allow_patterns=[f'{d}/*'])
        picked = [WORK/'hf_data'/d]
    # OpenOCR quét thư mục train bằng os.walk (KHÔNG theo symlink) -> dùng hardlink/copy thật
    import lmdb
    shutil.rmtree(TRAIN_ROOT, ignore_errors=True); TRAIN_ROOT.mkdir()
    for i, p in enumerate(picked):
        dst = TRAIN_ROOT/f'part{i}'; dst.mkdir()
        for f in Path(p).iterdir():
            if f.is_file():
                try: os.link(f, dst/f.name)
                except OSError: shutil.copy2(f, dst/f.name)
    leaf = [d_ for d_, dn, fn in os.walk(str(TRAIN_ROOT) + '/') if not dn]
    tot = 0
    for d_ in leaf:
        env = lmdb.open(d_, readonly=True, lock=False); tot += int(env.begin().get(b'num-samples')); env.close()
    print('thư mục LMDB train:', leaf, '| số mẫu train:', tot)
    assert tot >= BATCH, 'Không thấy dữ liệu train.'

In [ ]:
# ===== Dữ liệu tham chiếu của OpenOCR: IIIT5k gốc (kiểm tra baseline đọc đúng ~98%) + IC13 (chỉ để THEO DÕI lúc train, không chọn checkpoint) =====
from huggingface_hub import snapshot_download
snapshot_download(repo_id='topdu/OpenOCR-Data', repo_type='dataset', local_dir=WORK/'hf_eval', allow_patterns=['evaluation/IIIT5k/*', 'evaluation/IC13_857/*'])
IIIT_ORI = str(WORK/'hf_eval/evaluation/IIIT5k')
assert (Path(IIIT_ORI)/'data.mdb').exists(), 'Không tải được IIIT5k gốc từ HF (bật Internet trong Settings của Kaggle).'
VAL_DIR = WORK/'hf_eval/evaluation/IC13_857'
if not (VAL_DIR/'data.mdb').exists():
    VAL_DIR = Path(IIIT_ORI); print('CẢNH BÁO: không có IC13_857, theo dõi trên IIIT5k gốc (chỉ để theo dõi, tuyệt đối không chọn epoch/checkpoint theo số này)')
print('IIIT5k gốc:', IIIT_ORI, '| tập theo dõi:', VAL_DIR)


In [ ]:
assert tot==144677 and len(picked)==1, f'Expected Union14M medium 144677, got {tot} {picked}'; assert all(N[k]==3000 for k in ['clean','motion_mild','motion_medium','motion_severe'])

## 2. Checkpoint và cấu hình gốc

Kiểm SHA checkpoint/config chính thức. Chuyển checkpoint GTC+RCTC thành cùng cấu hình chỉ RCTC mà nhánh S2_AUG đã dùng; load `strict=True`. B0 phải cho số điểm của phép chạy trước trong sai số nhỏ, nếu không thì dữ liệu/config đã đổi.


In [ ]:
import hashlib,gdown,yaml,copy
CK=WORK/'official_tiny.pth';SV_CFG=WORK/'official_config.yml'
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
for path,gid,expect in [(CK,'1xJ10gOQphZjEgawXsgsadkXaZa0BIDN7','684f23bc3a2b1f724cadfd275e1a185456dd51b5e56c8830780d1998e90f77f6'),
 (SV_CFG,'1a-NRSh9aoYpdtwNgctF7a5CbEHcTA71a','f1ab35a0f5e44d3c24c940cdf1a07404068ad9191bad9fe232ad4fcf38bbbe29')]:
 local=LOCAL_TINY_CKPT if path==CK else LOCAL_TINY_CFG
 if local and not path.exists():shutil.copyfile(local,path)
 if not path.exists():gdown.download(id=gid,output=str(path),quiet=False)
 assert path.exists() and sha(path)==expect,f'Wrong official file {path}'
SV_CKPT=str(CK);SV_CFG=str(SV_CFG)
display(pd.DataFrame({'file':[SV_CKPT,SV_CFG],'sha256':[sha(CK),sha(SV_CFG)]}))


In [ ]:
from openrec.modeling import build_model
raw=torch.load(SV_CKPT,map_location='cpu',weights_only=False)
state={k.removeprefix('module.'):v for k,v in raw['state_dict'].items()}
mapped={};dropped=[]
for k,v in state.items():
 if k.startswith('encoder.'):mapped[k]=v
 elif k.startswith('decoder.ctc_decoder.'):mapped['decoder.'+k[len('decoder.ctc_decoder.'):]]=v
 elif k.startswith('decoder.gtc_decoder.'):dropped.append(k)
 else:raise ValueError('Unexpected checkpoint key: '+k)
arch=copy.deepcopy(yaml.safe_load(open(SV_CFG))['Architecture'])
assert arch['Encoder']['dims']==[64,128,256] and arch.get('Transform') is None
arch['algorithm']='SVTRv2';arch['Decoder']={'name':'RCTCDecoder','out_channels':mapped['decoder.fc.weight'].shape[0]}
model=build_model(arch);model.load_state_dict(mapped,strict=True)
assert sum(p.numel() for p in model.parameters())==5128255
S2_INIT=WORK/'s2_rctc_official_init.pth'
torch.save({'state_dict':model.state_dict()},S2_INIT)
print('S2 RCTC strict load OK; GTC keys removed',len(dropped),'init SHA',sha(S2_INIT))
del model,raw,state;torch.cuda.empty_cache()


In [ ]:
# ===== Tham số motion blur của nhóm -> thông số tăng cường (chỉ tổng hợp blur lên ẢNH TRAIN; không dùng ảnh test để train) =====
MBA_SRC = r'''import math
import os
import time

import cv2
import numpy as np
from PIL import Image


class MotionBlurAug:
    # Làm mờ chuyển động tuyến tính cho ảnh train (PIL RGB). Độ dài kernel tỉ lệ với chiều cao ảnh
    # để sau khi resize về 32/64 px độ mờ nhìn giống nhau với mọi cỡ ảnh gốc.

    def __init__(self, p=0.5, len_frac=(0.1, 0.5), angle=(0.0, 180.0), min_len=3, **kwargs):
        self.p = float(p)
        self.len_frac = (float(len_frac[0]), float(len_frac[1]))
        self.angle = (float(angle[0]), float(angle[1]))
        self.min_len = int(min_len)
        self._rng = None
        self._pid = None

    def _get_rng(self):
        if self._rng is None or self._pid != os.getpid():        # mỗi worker của DataLoader có luồng ngẫu nhiên riêng
            self._pid = os.getpid()
            self._rng = np.random.default_rng([self._pid, time.time_ns() % (2**32)])
        return self._rng

    @staticmethod
    def kernel(length, angle_deg):
        k = max(3, int(round(length)))
        if k % 2 == 0:
            k += 1
        ker = np.zeros((k, k), np.float32)
        c = k // 2
        a = math.radians(angle_deg)
        dx, dy = math.cos(a) * (k - 1) / 2.0, math.sin(a) * (k - 1) / 2.0
        cv2.line(ker, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))), 1.0, 1)
        s = float(ker.sum())
        if s <= 0:
            ker[c, c] = 1.0
            s = 1.0
        return ker / s

    def blur(self, arr, length, angle_deg):
        return cv2.filter2D(arr, -1, self.kernel(length, angle_deg), borderType=cv2.BORDER_REPLICATE)

    def __call__(self, data):
        rng = self._get_rng()
        if rng.random() >= self.p:
            return data
        arr = np.asarray(data['image'])
        length = rng.uniform(*self.len_frac) * arr.shape[0]
        if length < self.min_len:
            return data
        data['image'] = Image.fromarray(self.blur(arr, length, rng.uniform(*self.angle)))
        return data
'''
(OPENOCR/'openrec/preprocess/motion_blur_aug.py').write_text(MBA_SRC)
_p = OPENOCR/'openrec/preprocess/__init__.py'; _s = _p.read_text()
if "'MotionBlurAug'" not in _s: _p.write_text(_s.replace('MODULE_MAPPING = {', "MODULE_MAPPING = {\n    'MotionBlurAug': '.motion_blur_aug',", 1))
import cv2, importlib
importlib.invalidate_caches()
from openrec.preprocess.motion_blur_aug import MotionBlurAug



In [ ]:
# Dùng chính khoảng augmentation đã ghi trong S2_AUG_AUDIT.json, không điều chỉnh theo kết quả adapter.
LEN_FRAC=(0.052941176470588235,0.3176470588235294); ANGLE=(-10.,160.)
AUG_OP={'MotionBlurAug':dict(p=.5,len_frac=list(LEN_FRAC),angle=list(ANGLE))}
print('Fixed train augmentation',AUG_OP)


## 3. Sơ đồ gắn module và validation độc lập

Ở SVTRv2-T này, Stage 1 xuất 128 kênh, Stage 2/3 xuất 256 kênh. Stage 1/2 đã giảm kích thước bên trong; adapter nhận **đầu ra cuối stage** và trả lại cùng shape `(x, sz)`, để stage kế tiếp hoạt động đúng. Không chèn vào pixel và không sửa đầu đọc chữ.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(14,2.5));ax.set(xlim=(0,14),ylim=(0,2.5));ax.axis('off')
boxes=[('Ảnh',.2,'#dbeafe'),('Patch',1.9,'#e5e7eb'),('Stage 1',3.7,'#e5e7eb'),('A1',5.2,'#fde68a'),('Stage 2',6.4,'#e5e7eb'),('A2',7.9,'#fde68a'),('Stage 3',9.1,'#e5e7eb'),('A3',10.6,'#fde68a'),('RCTC',11.8,'#e5e7eb'),('Chữ',13,'#dbeafe')]
for name,x,col in boxes:
 w=.93 if name.startswith('A') else 1.15
 ax.add_patch(FancyBboxPatch((x,1.05),w,.65,boxstyle='round,pad=.03',facecolor=col,edgecolor='#475569'))
 ax.text(x+w/2,1.38,name,ha='center',va='center',fontsize=9)
for left,right in zip(boxes,boxes[1:]):
 x=left[1]+(.93 if left[0].startswith('A') else 1.15)
 ax.annotate('',xy=(right[1],1.38),xytext=(x,1.38),arrowprops={'arrowstyle':'->'})
ax.text(.2,.33,'Vàng: vị trí adapter có thể bật/tắt; xám: trọng số SVTRv2 gốc luôn cố định khi train adapter.',fontsize=10)
plt.tight_layout();plt.savefig(WORK/'pipeline_adapter_grid.png',dpi=160);plt.show()


In [ ]:
# Tạo IC13 sạch/blur cố định để xếp hạng pilot, KHÔNG dùng IIIT5K test để chọn.
val_pairs=read_lmdb_pairs(VAL_DIR)
assert 500<=len(val_pairs)<=2000, ('Unexpected validation source',len(val_pairs),VAL_DIR)
VAL_IMG=WORK/'val_images';VAL_IMG.mkdir(exist_ok=True)
clean_items=[];blur_items=[]
for i,(im,label) in enumerate(val_pairs):
 p=VAL_IMG/f'val_{i:05d}_clean.png';q=VAL_IMG/f'val_{i:05d}_motion.png'
 if not p.exists():im.save(p)
 if not q.exists():
  arr=np.asarray(im.convert('RGB')); rng=np.random.default_rng(2026+i)
  length=.25*arr.shape[0];angle=float(rng.uniform(-10,160))
  Image.fromarray(MotionBlurAug.blur(MotionBlurAug(p=1),arr,length,angle)).save(q)
 clean_items.append((p,label));blur_items.append((q,label))
VAL_LM=WORK/'val_fixed';VC=VAL_LM/'clean'/'test';VB=VAL_LM/'motion'/'test'
if not (VC/'data.mdb').exists():write_lmdb(clean_items,VC)
if not (VB/'data.mdb').exists():write_lmdb(blur_items,VB)
VAL_DIRS={'clean':str(VC),'motion':str(VB)}
(WORK/'val_dirs.json').write_text(json.dumps(VAL_DIRS,indent=2))
fig,axes=plt.subplots(2,4,figsize=(12,4))
for j in range(4):
 for i,path in enumerate((clean_items[j][0],blur_items[j][0])):
  axes[i,j].imshow(Image.open(path));axes[i,j].axis('off')
  if i==0:axes[i,j].set_title(clean_items[j][1])
plt.tight_layout();plt.savefig(WORK/'validation_pairs.png',dpi=150);plt.show()
print('Fixed validation images',len(val_pairs))


In [ ]:
# Cấu hình train/eval giữ nguyên transforms, sampler, CTC từ S2_AUG; thay riêng model trainable trong worker.
import yaml,copy
RCTC_REF=yaml.safe_load(open(OPENOCR/'configs/rec/svtrv2/svtrv2_tiny_rctc.yml'))
BASE_CFG=yaml.safe_load(open(SV_CFG))
TRAIN_DIRS=[str(d) for d,sub,files in os.walk(str(TRAIN_ROOT)+'/') if not sub and (Path(d)/'data.mdb').exists()]
assert TRAIN_DIRS and len(TRAIN_DIRS)==1
c=copy.deepcopy(BASE_CFG)
c['Architecture']=copy.deepcopy(arch);c['Architecture']['Decoder'].pop('out_channels')
for key in ('Loss','PostProcess','Metric'):c[key]=copy.deepcopy(RCTC_REF[key])
c['Global'].update(seed=SEED,pretrained_model=str(S2_INIT),checkpoints=None)
c['Train']['dataset']['data_dir_list']=TRAIN_DIRS;c['Train']['dataset'].pop('ratio_list',None)
c['Train']['dataset']['transforms']=[{'DecodeImagePIL':{'img_mode':'RGB'}},{'PARSeqAugPIL':None},AUG_OP,
 {'CTCLabelEncode':{'character_dict_path':str(OPENOCR/'tools/utils/EN_symbol_dict.txt'),'use_space_char':False,'max_text_length':25}},
 {'KeepKeys':{'keep_keys':['image','label','length']}}]
c['Train']['sampler']['first_bs']=48;c['Train']['loader'].update(batch_size_per_card=48,num_workers=2)
c['Eval']['dataset']['transforms']=[c['Train']['dataset']['transforms'][0],c['Train']['dataset']['transforms'][-2],c['Train']['dataset']['transforms'][-1]]
c['Eval']['loader'].update(batch_size_per_card=128,num_workers=2)
c['Eval']['dataset']['data_dir_list']=[str(VC)]
CFG=WORK/'adapter_grid.yml';CFG.write_text(yaml.safe_dump(c,sort_keys=False))
print('Train data',TRAIN_DIRS,'augmentation',c['Train']['dataset']['transforms'][2])


## 4. Worker và kiểm tra trước train

Script được đóng trong notebook và ghi ra `/kaggle/working`. Trước train, mỗi cấu hình kiểm: checkpoint load strict, shape từng stage, adapter là identity lúc khởi tạo, gradient chỉ ở adapter, loss hữu hạn. Bất cứ sai khác nào dừng ngay. Mỗi nhánh dùng checkpoint B0 giống nhau, seed giống nhau và đúng số bước.


In [ ]:
ADAPTER_WORKER = '"""Frozen SVTRv2-T/RCTC with post-stage residual adapters; Kaggle worker.\n\nThe stage hooks replace (feature, spatial_size) with (adapter(feature), spatial_size).\nThe OpenOCR encoder and decoder remain frozen. This is an experiment designed for\nthis project, not a reproduction of an adapter paper.\n"""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\nclass ResidualAdapter(nn.Module):\n    """Identity at initialization; 1x5 spatial depthwise + bottleneck mixing."""\n\n    def __init__(self, channels, reduction=4):\n        super().__init__()\n        hidden = max(16, channels // reduction)\n        self.norm = nn.GroupNorm(1, channels)\n        self.dw = nn.Conv2d(channels, channels, (1, 5), padding=(0, 2), groups=channels)\n        self.down = nn.Conv2d(channels, hidden, 1)\n        self.act = nn.GELU()\n        self.up = nn.Conv2d(hidden, channels, 1)\n        nn.init.zeros_(self.up.weight)\n        nn.init.zeros_(self.up.bias)\n\n    def forward(self, x, sz):\n        sequence = x.ndim == 3\n        if sequence:\n            b, n, c = x.shape\n            assert n == int(sz[0]) * int(sz[1]), (x.shape, sz)\n            y = x.transpose(1, 2).reshape(b, c, int(sz[0]), int(sz[1]))\n        else:\n            assert x.ndim == 4 and tuple(x.shape[-2:]) == tuple(sz), (x.shape, sz)\n            y = x\n        delta = self.up(self.act(self.down(self.dw(self.norm(y)))))\n        if sequence:\n            delta = delta.flatten(2).transpose(1, 2)\n        return x + delta\n\n\nclass HookedOCR(nn.Module):\n    def __init__(self, base, selected, channels):\n        super().__init__()\n        self.base = base\n        self.selected = tuple(selected)\n        self.adapters = nn.ModuleDict({str(i): ResidualAdapter(channels[i]) for i in selected})\n        self.before = {}\n        self.after = {}\n        self.handles = []\n        for i in range(3):\n            self.handles.append(self.base.encoder.stages[i].register_forward_hook(self._hook(i)))\n\n    def _hook(self, index):\n        def apply(_module, _args, output):\n            assert isinstance(output, tuple) and len(output) == 2\n            x, sz = output\n            self.before[index] = x.detach()\n            if str(index) in self.adapters:\n                x = self.adapters[str(index)](x, sz)\n            self.after[index] = x.detach()\n            return x, sz\n        return apply\n\n    def forward(self, x):\n        self.before.clear()\n        self.after.clear()\n        return self.base(x)\n\n\ndef load_base(cfg, checkpoint, device):\n    from openrec.modeling import build_model\n    from openrec.postprocess import build_post_process\n    from openrec.losses import build_loss\n    from openrec.metrics import build_metric\n    post = build_post_process(cfg["PostProcess"], cfg["Global"])\n    arch = copy.deepcopy(cfg["Architecture"])\n    arch["Decoder"]["out_channels"] = post.get_character_num()\n    base = build_model(arch)\n    raw = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    state = raw["state_dict"] if "state_dict" in raw else raw\n    state = {k.removeprefix("module."): v for k, v in state.items()}\n    base.load_state_dict(state, strict=True)\n    base.to(device).eval().requires_grad_(False)\n    assert len(base.encoder.stages) == 4 and not tuple(base.encoder.stages[3].parameters())\n    return base, post, build_loss(cfg["Loss"]), build_metric(cfg["Metric"])\n\n\ndef norm(metric, value):\n    if metric.ignore_space:\n        value = value.replace(" ", "")\n    if metric.is_filter:\n        value = metric._normalize_text(value)\n    if metric.is_lower:\n        value = value.lower()\n    return value\n\n\n@torch.no_grad()\ndef evaluate(net, post, metric, cfg, data_dir, device, logger, save_path=None):\n    from tools.data import build_dataloader\n    c = copy.deepcopy(cfg)\n    c["Eval"]["dataset"]["data_dir_list"] = [str(data_dir)]\n    random.seed(991); np.random.seed(991); torch.manual_seed(991)\n    loader = build_dataloader(c, "Eval", logger)\n    net.eval(); metric.reset()\n    targets, preds, correct = [], [], []\n    for batch in loader:\n        probs = net(batch[0].to(device))\n        pred_batch, target_batch = post(probs, [x.numpy() for x in batch])\n        metric((pred_batch, target_batch))\n        for (p, _), (t, _) in zip(pred_batch, target_batch):\n            preds.append(p); targets.append(t); correct.append(norm(metric, p) == norm(metric, t))\n    acc = sum(correct) / len(correct)\n    assert abs(acc - metric.get_metric()["acc"]) < 1e-4\n    result = dict(acc=acc, n=len(targets), targets=targets, preds=preds, correct=correct)\n    if save_path is not None:\n        Path(save_path).write_text(json.dumps(result, ensure_ascii=False))\n    return result\n\n\ndef construct(args, cfg, device):\n    base, post, loss_fn, metric = load_base(cfg, args.init, device)\n    channels = {0: 128, 1: 256, 2: 256}  # official Tiny config; checked below by a forward pass\n    assert cfg["Architecture"]["Encoder"]["dims"] == [64, 128, 256]\n    selected = [int(v) - 1 for v in args.placements.split("+")]\n    assert selected == sorted(set(selected)) and all(i in (0, 1, 2) for i in selected)\n    net = HookedOCR(base, selected, channels).to(device)\n    return net, post, loss_fn, metric\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    for key in ("openocr", "cfg", "init", "placements", "out", "val_dirs"):\n        p.add_argument("--" + key, required=True)\n    p.add_argument("--epochs", type=int, default=2)\n    p.add_argument("--seed", type=int, default=1234)\n    p.add_argument("--lr", type=float, default=2e-4)\n    p.add_argument("--eval_dirs", default="")\n    p.add_argument("--smoke", action="store_true")\n    args = p.parse_args()\n    import yaml\n    sys.path.insert(0, str(Path(args.openocr).resolve()))\n    os.chdir(args.openocr)\n    from tools.data import build_dataloader\n    from tools.utils.logging import get_logger\n    logger = get_logger()\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    assert device.type == "cuda", "Training requires Kaggle GPU"\n    cfg = yaml.safe_load(Path(args.cfg).read_text())\n    net, post, loss_fn, metric = construct(args, cfg, device)\n    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    trainable = [v for v in net.adapters.parameters() if v.requires_grad]\n    assert trainable and all(not x.requires_grad for x in net.base.parameters())\n    train_loader = build_dataloader(cfg, "Train", logger)\n    batch = next(iter(train_loader))\n    x = batch[0].to(device)\n    net.base.eval(); net.base.decoder.train()\n    logits = net(x)\n    assert logits.shape[0] == x.shape[0] and bool(torch.isfinite(logits).all())\n    shapes = {}\n    for i in range(3):\n        y = net.before[i]\n        ch = y.shape[1] if y.ndim == 4 else y.shape[-1]\n        assert ch == {0: 128, 1: 256, 2: 256}[i], (i, y.shape)\n        shapes[str(i + 1)] = list(y.shape)\n    for i in net.selected:\n        assert torch.allclose(net.before[i], net.after[i], atol=1e-6), i\n    loss = loss_fn(logits.float(), [x, batch[1].to(device), batch[2].to(device)])["loss"]\n    assert torch.isfinite(loss), loss\n    loss.backward()\n    assert any(v.grad is not None and torch.isfinite(v.grad).all() and bool((v.grad != 0).any()) for v in trainable)\n    assert all(v.grad is None for v in net.base.parameters())\n    smoke = dict(shapes=shapes, loss=float(loss.detach()), trainable=sum(v.numel() for v in trainable),\n                 base_frozen=True, identity_at_init=True, gradient_ok=True, input_shape=list(x.shape))\n    (out / "smoke.json").write_text(json.dumps(smoke, indent=2))\n    print("SMOKE", smoke, flush=True)\n    if args.smoke:\n        return\n    net.zero_grad(set_to_none=True)\n    steps_per_epoch = len(train_loader)\n    total = args.epochs * steps_per_epoch\n    opt = torch.optim.AdamW(trainable, lr=args.lr, weight_decay=0.05)\n    warm = max(1, int(0.05 * total))\n    def factor(step):\n        if step < warm:\n            return (step + 1) / warm\n        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, factor)\n    latest = out / "latest.pth"\n    start_epoch = 0; step = 0\n    if latest.exists():\n        saved = torch.load(latest, map_location="cpu", weights_only=False)\n        assert saved["init_sha"] == sha256(args.init) and saved["cfg_sha"] == sha256(args.cfg)\n        assert saved["placements"] == args.placements\n        assert saved["epochs"] == args.epochs and saved["seed"] == args.seed\n        net.adapters.load_state_dict(saved["adapters"], strict=True)\n        opt.load_state_dict(saved["optimizer"]); scheduler.load_state_dict(saved["scheduler"])\n        start_epoch = saved["epoch"]; step = saved["step"]\n    with (out / "train_log.csv").open("a", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=["epoch", "step", "ctc", "lr"])\n        if f.tell() == 0:\n            writer.writeheader()\n        for epoch in range(start_epoch, args.epochs):\n            if hasattr(train_loader.batch_sampler, "set_epoch"):\n                train_loader.batch_sampler.set_epoch(args.seed * 1000 + epoch)\n            net.base.eval(); net.base.decoder.train(); net.adapters.train()\n            losses = []\n            for batch in train_loader:\n                xx = batch[0].to(device)\n                pred = net(xx)\n                val = loss_fn(pred.float(), [xx, batch[1].to(device), batch[2].to(device)])["loss"]\n                assert torch.isfinite(val), (epoch, step, val)\n                opt.zero_grad(set_to_none=True)\n                val.backward()\n                torch.nn.utils.clip_grad_norm_(trainable, max_norm=5.0)\n                opt.step(); scheduler.step(); step += 1\n                losses.append(float(val.detach()))\n                if step % 200 == 0:\n                    print(args.placements, epoch + 1, step, "loss", round(np.mean(losses[-200:]), 4), flush=True)\n            writer.writerow(dict(epoch=epoch + 1, step=step, ctc=float(np.mean(losses)), lr=opt.param_groups[0]["lr"]))\n            f.flush()\n            torch.save(dict(adapters=net.adapters.state_dict(), optimizer=opt.state_dict(), scheduler=scheduler.state_dict(),\n                            epoch=epoch + 1, step=step, epochs=args.epochs, seed=args.seed,\n                            placements=args.placements, init_sha=sha256(args.init), cfg_sha=sha256(args.cfg)), latest)\n    # Validation is separate from the 3000-image IIIT5K test; no test-based checkpoint selection.\n    valid = {}\n    for condition, data_dir in json.loads(Path(args.val_dirs).read_text()).items():\n        valid[condition] = evaluate(net, post, metric, cfg, data_dir, device, logger)["acc"] * 100\n    (out / "validation.json").write_text(json.dumps(valid, indent=2))\n    if args.eval_dirs:\n        for condition, data_dir in json.loads(Path(args.eval_dirs).read_text()).items():\n            evaluate(net, post, metric, cfg, data_dir, device, logger, out / f"persample_{condition}.json")\n    print("DONE", args.placements, args.epochs, valid, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
SCRIPT=WORK/'adapter_grid_trial.py';SCRIPT.write_text(ADAPTER_WORKER)
import importlib.util
spec=importlib.util.spec_from_file_location('adapter_grid_trial',SCRIPT)
G=importlib.util.module_from_spec(spec);spec.loader.exec_module(G)
GRID=['1','2','3','1+2','1+3','2+3','1+2+3']
import subprocess
for name in GRID:
 env={**os.environ,'CUDA_VISIBLE_DEVICES':'0','PYTHONPATH':str(OPENOCR)}
 cmd=[sys.executable,str(SCRIPT),'--openocr',str(OPENOCR),'--cfg',str(CFG),'--init',str(S2_INIT),
  '--placements',name,'--out',str(WORK/'pilot'/name),'--val_dirs',str(WORK/'val_dirs.json'),'--epochs',str(PILOT_EPOCHS),'--smoke']
 r=subprocess.run(cmd,cwd=OPENOCR,env=env,capture_output=True,text=True)
 print(name,'smoke exit',r.returncode,(r.stdout+r.stderr)[-1200:]);assert r.returncode==0,name


## 5. Pilot 7 tổ hợp: mỗi tổ hợp 2 epoch, hai GPU chạy đồng thời

Điểm pilot chỉ dùng để chọn trên IC13 cố định. Không xem IIIT5K để chọn. Một GPU xử lý một tổ hợp tại một thời điểm. Log và checkpoint lưu sau mỗi epoch để tiếp tục được khi session còn file.


In [ ]:
import time
def run_jobs(names,epochs,root,eval_file=''):
 pending=list(names);active=[];done=[]
 while pending or active:
  while pending and len(active)<2:
   name=pending.pop(0);gpu=next(str(i) for i in range(2) if str(i) not in {a[1] for a in active});out=WORK/root/name;out.mkdir(parents=True,exist_ok=True)
   cmd=[sys.executable,str(SCRIPT),'--openocr',str(OPENOCR),'--cfg',str(CFG),'--init',str(S2_INIT),
    '--placements',name,'--out',str(out),'--val_dirs',str(WORK/'val_dirs.json'),
    '--epochs',str(epochs),'--seed',str(SEED),'--lr','0.0002']
   if eval_file:cmd+=['--eval_dirs',str(eval_file)]
   handle=open(out/'process.log','a')
   proc=subprocess.Popen(cmd,cwd=OPENOCR,env={**os.environ,'CUDA_VISIBLE_DEVICES':gpu,'PYTHONPATH':str(OPENOCR)},stdout=handle,stderr=subprocess.STDOUT)
   active.append((name,gpu,proc,handle,out))
  time.sleep(20)
  for item in active[:]:
   name,gpu,proc,handle,out=item
   if proc.poll() is not None:
    handle.close();active.remove(item);print(name,'GPU',gpu,'exit',proc.returncode)
    if proc.returncode:print((out/'process.log').read_text(errors='replace')[-5000:])
    assert proc.returncode==0,name
    saved=torch.load(out/'latest.pth',map_location='cpu',weights_only=False)
    assert saved['epoch']==epochs and saved['step']==epochs*3192,(name,saved['epoch'],saved['step'])
    done.append(name)
  print('Completed',len(done),'/',len(names),'active',[a[0] for a in active])
 return done
run_jobs(GRID,PILOT_EPOCHS,'pilot')


In [ ]:
pilot=[]
for name in GRID:
 p=WORK/'pilot'/name;v=json.loads((p/'validation.json').read_text())
 smoke=json.loads((p/'smoke.json').read_text())
 pilot.append(dict(placement=name,clean=v['clean'],motion=v['motion'],params=smoke['trainable']))
PILOT=pd.DataFrame(pilot).sort_values(['motion','clean'],ascending=False).reset_index(drop=True)
PILOT.to_csv(WORK/'pilot_validation.csv',index=False);display(PILOT)
TOP=PILOT['placement'].head(TOP_K).tolist();print('Top two selected ONLY by fixed IC13 validation:',TOP)
fig,ax=plt.subplots(figsize=(10,4));PILOT.set_index('placement')[['clean','motion']].plot.bar(ax=ax)
ax.set_ylabel('Word accuracy IC13 validation (%)');plt.tight_layout();plt.savefig(WORK/'pilot_validation.png',dpi=150);plt.show()


## 6. Train lại từ B0 đủ 12 epoch cho 2 tổ hợp đã chọn

Không nối checkpoint pilot vì pilot có lịch LR 2 epoch; train lại từ đầu bảo đảm lịch 12 epoch thống nhất. Mỗi GPU nhận một tổ hợp. S2_AUG tham chiếu cũng được train 12 epoch trong notebook trước.


In [ ]:
run_jobs(TOP,FINAL_EPOCHS,'final')

In [ ]:
# Theo dõi loss từng epoch để bắt run không học/NaN, không dùng test để chọn checkpoint.
fig,axes=plt.subplots(1,2,figsize=(12,3.5))
for name in GRID:
 log=pd.read_csv(WORK/'pilot'/name/'train_log.csv')
 assert len(log)==PILOT_EPOCHS and np.isfinite(log['ctc']).all()
 axes[0].plot(log['epoch'],log['ctc'],marker='o',label=name)
for name in TOP:
 log=pd.read_csv(WORK/'final'/name/'train_log.csv')
 assert len(log)==FINAL_EPOCHS and np.isfinite(log['ctc']).all()
 axes[1].plot(log['epoch'],log['ctc'],marker='.',label=name)
for ax,title in zip(axes,['Pilot 2 epoch','Final 12 epoch']):ax.set(title=title,xlabel='Epoch',ylabel='Train CTC loss');ax.legend(fontsize=7)
plt.tight_layout();plt.savefig(WORK/'training_curves.png',dpi=150);plt.show()


## 7. Đánh giá test một lần, cùng ảnh và cùng nhãn

Tìm checkpoint S2_AUG bạn đã train; kiểm cấu trúc, sau đó đo lại B0, S2_AUG và hai adapter cuối trên bốn điều kiện. Xuất accuracy, CER, số từ sửa đúng/làm sai và paired bootstrap cho từng điều kiện. Không dùng các số test này để chọn lại tổ hợp.


In [ ]:
import hashlib
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
if not S2_AUG_CKPT:
 hits=list(Path('/kaggle/input').rglob('S2_AUG/latest.pth'))
 assert len(hits)==1, f'Need exactly one S2_AUG/latest.pth, found {hits}. Set S2_AUG_CKPT.'
 S2_AUG_CKPT=str(hits[0])
assert Path(S2_AUG_CKPT).exists(), 'Add Kaggle output of prior S2_AUG run (audit-only ZIP lacks weights)'
print('S2_AUG reference',S2_AUG_CKPT,'SHA',sha(S2_AUG_CKPT))
TEST_DIRS={k:str(LM/k/'test') for k in ['clean','motion_mild','motion_medium','motion_severe']}
(WORK/'test_dirs.json').write_text(json.dumps(TEST_DIRS,indent=2))


In [ ]:
from tools.utils.logging import get_logger
logger=get_logger();dev=torch.device('cuda:0')
MODELS={'B0':str(S2_INIT),'S2_AUG':str(S2_AUG_CKPT)}
REF={}
for name,checkpoint in MODELS.items():
 base,post,_,metric=G.load_base(c,checkpoint,dev)
 net=G.HookedOCR(base,[],{0:128,1:256,2:256}).to(dev)
 REF[name]={}
 for cond,d in TEST_DIRS.items():
  dest=WORK/f'persample_{name}_{cond}.json'
  REF[name][cond]=G.evaluate(net,post,metric,c,d,dev,logger,dest)
 del net,base;torch.cuda.empty_cache()
print('B0', {k:round(v['acc']*100,2) for k,v in REF['B0'].items()})
print('S2_AUG', {k:round(v['acc']*100,2) for k,v in REF['S2_AUG'].items()})
assert abs(100*REF['B0']['clean']['acc']-98.60)<.2 and abs(100*REF['B0']['motion_severe']['acc']-80.23)<.2,'B0 protocol mismatch'
assert abs(100*REF['S2_AUG']['motion_severe']['acc']-89.17)<.3,'S2_AUG differs from audited previous run; inspect checkpoint/config/data'


In [ ]:
# Evaluate final adapters without retraining; code used to train/eval is identical.
for name in TOP:
 out=WORK/'final'/name;ck=torch.load(out/'latest.pth',map_location='cpu',weights_only=False)
 assert ck['epoch']==FINAL_EPOCHS and ck['init_sha']==sha(S2_INIT)
 args=type('A',(),{'init':str(S2_INIT),'placements':name})()
 net,post,_,metric=G.construct(args,c,dev)
 net.adapters.load_state_dict(ck['adapters'],strict=True)
 REF['A'+name]={}
 for cond,d in TEST_DIRS.items():
  REF['A'+name][cond]=G.evaluate(net,post,metric,c,d,dev,logger,WORK/f'persample_A{name}_{cond}.json')
 del net;torch.cuda.empty_cache()
print('Final evaluation complete:',list(REF))


In [ ]:
from rapidfuzz.distance.Levenshtein import distance
rows=[];pair=[];rng=np.random.default_rng(20261118)
for name,by_cond in REF.items():
 for cond,r in by_cond.items():
  cer=sum(distance(G.norm(metric,p),G.norm(metric,t)) for p,t in zip(r['preds'],r['targets']))/sum(len(G.norm(metric,t)) for t in r['targets'])*100
  rows.append(dict(model=name,condition=cond,n=r['n'],word_acc=100*r['acc'],cer=cer))
  if name!='B0':
   a=np.array(REF['B0'][cond]['correct'],bool);b=np.array(r['correct'],bool)
   assert REF['B0'][cond]['targets']==r['targets'] and len(a)==3000
   delta=b.astype(np.int8)-a.astype(np.int8)
   boots=np.array([delta[rng.integers(0,len(delta),len(delta))].mean()*100 for _ in range(2000)])
   pair.append(dict(reference='B0',model=name,condition=cond,fixed=int((~a&b).sum()),broken=int((a&~b).sum()),delta_pp=float(delta.mean()*100),ci_low=float(np.quantile(boots,.025)),ci_high=float(np.quantile(boots,.975))))
RESULTS=pd.DataFrame(rows);PAIRED=pd.DataFrame(pair)
display(RESULTS.pivot(index='model',columns='condition',values='word_acc').round(2));display(RESULTS.pivot(index='model',columns='condition',values='cer').round(2));display(PAIRED)
RESULTS.to_csv(WORK/'final_accuracy_cer.csv',index=False);PAIRED.to_csv(WORK/'final_paired_vs_B0.csv',index=False)


In [ ]:
# Đếm sửa đúng/làm sai trực tiếp so với S2_AUG, không chỉ B0.
vs_s2=[]
for name in ['A'+x for x in TOP]:
 for cond in TEST_DIRS:
  a=np.array(REF['S2_AUG'][cond]['correct'],bool);b=np.array(REF[name][cond]['correct'],bool)
  assert REF['S2_AUG'][cond]['targets']==REF[name][cond]['targets']
  vs_s2.append(dict(model=name,condition=cond,fixed=int((~a&b).sum()),broken=int((a&~b).sum()),delta_pp=100*(b.mean()-a.mean())))
VS_S2=pd.DataFrame(vs_s2);display(VS_S2);VS_S2.to_csv(WORK/'final_paired_vs_S2_AUG.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,4))
for j,col in enumerate(['word_acc','cer']):
 RESULTS.pivot(index='condition',columns='model',values=col).plot.bar(ax=axes[j]);axes[j].set_ylabel(col);axes[j].legend(fontsize=7)
plt.tight_layout();plt.savefig(WORK/'final_scores.png',dpi=160);plt.show()


In [ ]:
# Ví dụ chữ thật/dự đoán: lấy từ đúng cùng vị trí trong 3.000 ảnh test.
examples=[]
for name in ['A'+x for x in TOP]:
 for cond in ['motion_medium','motion_severe']:
  a=REF['S2_AUG'][cond];b=REF[name][cond]
  for kind,mask in [('fixed',[(not x) and y for x,y in zip(a['correct'],b['correct'])]),
                    ('broken',[x and (not y) for x,y in zip(a['correct'],b['correct'])])]:
   ix=[i for i,v in enumerate(mask) if v][:8]
   for i in ix:examples.append(dict(model=name,condition=cond,kind=kind,index_in_eval_order=i,
      target=a['targets'][i],s2_prediction=a['preds'][i],adapter_prediction=b['preds'][i]))
EXAMPLES=pd.DataFrame(examples);display(EXAMPLES);EXAMPLES.to_csv(WORK/'word_examples.csv',index=False)


## 8. Nhìn feature thực tế: chỗ nào adapter thay đổi, có liên quan đến từ đọc sai không?

Hình đầu tiên cho cường độ phần chỉnh feature ở A1/A2/A3 trên **cùng một batch blur**. Hình thứ hai cho khoảng cách feature blur so với ảnh sạch tại từng stage. Đây là phép quan sát, không coi feature gần ảnh sạch hơn là bằng chứng OCR sẽ tăng. Bảng đúng/sai ở trên mới là tiêu chí chính.


In [ ]:
from tools.data import build_dataloader
def first_batch(condition):
 cc=copy.deepcopy(c);cc['Eval']['dataset']['data_dir_list']=[TEST_DIRS[condition]]
 random.seed(991);np.random.seed(991);torch.manual_seed(991)
 return next(iter(build_dataloader(cc,'Eval',logger)))
bc=first_batch('clean');bb=first_batch('motion_severe')
same_batch=bc[0].shape==bb[0].shape and torch.equal(bc[1],bb[1])
print('Clean/blur batch alignment:',same_batch,'shapes',tuple(bc[0].shape),tuple(bb[0].shape))
assert same_batch,'Cannot compare feature tensors by sample; inspect ratio sampler/order'
feature_rows=[];maps=[]
for name in ['B0']+['A'+x for x in TOP]:
 checkpoint=S2_INIT
 args=type('A',(),{'init':str(checkpoint),'placements':name[1:] if name!='B0' else '1'})()
 net,post,_,metric=G.construct(args,c,dev)
 if name=='B0':net.adapters= torch.nn.ModuleDict()  # stage hooks stay, no corrections
 else:
  ck=torch.load(WORK/'final'/name[1:]/'latest.pth',map_location='cpu',weights_only=False)
  net.adapters.load_state_dict(ck['adapters'],strict=True)
 net.eval()
 with torch.no_grad():
  _=net(bc[0].to(dev));clean_feat={i:t.float().cpu() for i,t in net.after.items()}
  _=net(bb[0].to(dev));blur_feat={i:t.float().cpu() for i,t in net.after.items()}
  before={i:t.float().cpu() for i,t in net.before.items()}
 for i in range(3):
  a=blur_feat[i];z=clean_feat[i];d=a-before[i]
  feature_rows.append(dict(model=name,stage=i+1,blur_clean_rel_l2=float((a-z).norm()/(z.norm()+1e-8)),adapter_delta_rel_l2=float(d.norm()/(before[i].norm()+1e-8))))
  if name!='B0' and i in [int(v)-1 for v in name[1:].split('+')]:
   heat=d[0].abs().mean(0) if d.ndim==4 else d[0].abs().mean(-1)
   maps.append((name,i+1,heat.numpy()))
 del net;torch.cuda.empty_cache()
FEATURE=pd.DataFrame(feature_rows);display(FEATURE);FEATURE.to_csv(WORK/'feature_probes.csv',index=False)
fig,axes=plt.subplots(len(maps),1,figsize=(10,max(2,1.6*len(maps))),squeeze=False)
for ax,(name,stage,heat) in zip(axes[:,0],maps):
 ax.imshow(np.atleast_2d(heat),aspect='auto',cmap='magma');ax.set_ylabel(f'{name} S{stage}')
plt.tight_layout();plt.savefig(WORK/'adapter_feature_maps.png',dpi=150);plt.show()


## 9. Audit và gói gửi

Gói nhẹ chứa config, source, log, bảng theo ảnh, hình pipeline/feature và hash checkpoint. Nó **không chứa trọng số**; lưu riêng hai file `final/<name>/latest.pth` và checkpoint S2_AUG nếu muốn chạy lại. Notebook không khẳng định adapter thắng nếu accuracy/CER không cùng tăng. Nếu tăng trên IIIT5K nhưng giảm trên IC13 hoặc blur mới, báo cả hai.


In [ ]:
import zipfile
AUDIT=dict(openocr_commit=subprocess.check_output(['git','-C',str(OPENOCR),'rev-parse','HEAD'],text=True).strip(),
 official_checkpoint_sha=sha(CK),official_config_sha=sha(SV_CFG),s2_initial_sha=sha(S2_INIT),
 s2_aug_reference_sha=sha(S2_AUG_CKPT),train_lmdb_sha=[sha(Path(x)/'data.mdb') for x in TRAIN_DIRS],
 train_n=tot,train_aug=AUG_OP,pilot_epochs=PILOT_EPOCHS,final_epochs=FINAL_EPOCHS,pilot_grid=GRID,selected_on_ic13=TOP,
 smoke={name:json.loads((WORK/'pilot'/name/'smoke.json').read_text()) for name in GRID},
 final={name:dict(checkpoint_sha=sha(WORK/'final'/name/'latest.pth'),validation=json.loads((WORK/'final'/name/'validation.json').read_text())) for name in TOP},
 limitations=['Blur range derived from previously inspected IIIT5K protocol','Union14M/IIIT5K content overlap not exhaustively audited','MotionBlurAug RNG from worker PID/time','IC13 validation synthetic motion is not an independent real-world blur set','Adapter is a project-specific design, not paper reproduction'])
(WORK/'adapter_grid_audit.json').write_text(json.dumps(AUDIT,ensure_ascii=False,indent=2))
files=[CFG,SCRIPT,WORK/'adapter_grid_audit.json',WORK/'pilot_validation.csv',WORK/'final_accuracy_cer.csv',
 WORK/'final_paired_vs_B0.csv',WORK/'final_paired_vs_S2_AUG.csv',WORK/'feature_probes.csv',
 WORK/'word_examples.csv',WORK/'pipeline_adapter_grid.png',WORK/'validation_pairs.png',WORK/'pilot_validation.png',WORK/'training_curves.png',WORK/'final_scores.png',WORK/'adapter_feature_maps.png']
files+=list(WORK.glob('persample_*.json'))
for name in GRID:files += [WORK/'pilot'/name/'smoke.json',WORK/'pilot'/name/'validation.json',WORK/'pilot'/name/'train_log.csv',WORK/'pilot'/name/'process.log']
for name in TOP:files += [WORK/'final'/name/'smoke.json',WORK/'final'/name/'validation.json',WORK/'final'/name/'train_log.csv',WORK/'final'/name/'process.log']
package=Path('/kaggle/working/Danh_adapter_grid_audit_only.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
 for p in files:
  assert p.exists(),p
  z.write(p,str(p.relative_to(WORK)))
print('Send:',package,'MB',round(package.stat().st_size/1e6,2))
print('Keep trained checkpoints separately:',[str(WORK/'final'/name/'latest.pth') for name in TOP])
